In [ ]:
import pandas as pd
import string
from collections import Counter
from nltk.util import ngrams
from nltk.corpus import stopwords
import spacy

# Load spaCy models
nlp_en = spacy.load("en_core_web_sm")
nlp_es = spacy.load("es_core_news_sm")

# Load raw data into a data frame
df_raw = pd.read_csv("C:/Users/HP/OneDrive/Desktop/CUCEA 2025 B/2do Semestre/Programacion II/proyecto_challenge/Basico/data/reviews_microsoft.csv")
df_raw.head()

# --- 1. DATE CLEANING & SELECTION ---
df = df_raw[['location','dates','job-title', 'summary','pros','cons', 'overall-ratings']].copy()
df['dates'] = pd.to_datetime(df['dates'], errors='coerce')
df = df.dropna().head(1000)

# --- 2. CREATE ENGLISH-SPANISH DATA FRAME --- 
df_en = df.copy()
df_en['language'] = 'en'
df_es = df.copy()
df_es['summary'] = "El trabajo es muy estresante y no hay comunicación" # More realistic simulation
df_es['language'] = 'es'

# --- 3. DATA CLEANING FUNCTION ---
def clean_text(text):
    text = str(text).lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = "".join([i for i in text if not i.isdigit()])
    return text

# --- 4. IMPROVED STOPWORDS & LEMMATIZATION ---
def process_corpus(text, lang='en'):
    if lang == 'en':
        doc = nlp_en(text)
        # Custom English Stopwords
        stops = set(stopwords.words('english')) - {'not', 'no', 'won', 'dont', 'isnt'}
    else:
        doc = nlp_es(text)
        # Custom Spanish Stopwords
        stops = set(stopwords.words('spanish')) - {'no', 'ni', 'poco', 'nada'}
    
    # Process lemmas while filtering custom stopwords
    lemmas = [token.lemma_ for token in doc if token.text not in stops and not token.is_space]
    return lemmas

# Applying the logic
df_en['cleaned'] = df_en['summary'].apply(clean_text)
df_es['cleaned'] = df_es['summary'].apply(clean_text)

df_en['lemmas'] = df_en['cleaned'].apply(lambda x: process_corpus(x, lang='en'))
df_es['lemmas'] = df_es['cleaned'].apply(lambda x: process_corpus(x, lang='es'))

# --- 5. N-GRAMS DISTRIBUTIONS ---
def get_ngrams(lemmas_list, n=2):
    all_words = [word for sublist in lemmas_list for word in sublist]
    n_grams = ngrams(all_words, n)
    return Counter(n_grams).most_common(10)

# Example Output
print("Top English Bigrams:", get_ngrams(df_en['lemmas'], 2))


# --- 6. SAVE PROCESSED DATA ---
# Ensure the 'data' directory exists
import os
if not os.path.exists('data'):
    os.makedirs('data')

# The SVM and VADER scripts need strings, not lists. 
# We join the lemmas back into a clean sentence.
df_en['lemmas_joined'] = df_en['lemmas'].apply(lambda x: " ".join(x))

# Save the English data for the SVM trainer
df_en.to_csv("data/processed_en.csv", index=False)

print("✅ Data cleaning complete. Files saved to /data/ folder.")